# CardioSense: Game-Theoretic SHAP Explainability
**Cohort**: Primary Indian Hospital Cohort ($N=1,000$)
**Model**: Champion XGBoost Pipeline
**Methodology**: Shapley Additive exPlanations (`shap.TreeExplainer`) satisfying the Axioms of Efficiency, Symmetry, Dummy Player, and Additivity.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import get_dataset_config, MODELS_DIR, FIGURES_DIR
from src.preprocessing import get_feature_names_out


## 1. Load Champion Pipeline and Test Data


In [ ]:
cfg = get_dataset_config("india")
model_path = MODELS_DIR / "india" / "best_pipeline.joblib"
pipeline = joblib.load(model_path)

test_df = pd.read_csv(cfg["test_file"])
X_test = test_df[cfg["feature_names"]]
y_test = test_df[cfg["target_column"]]

preprocessor = pipeline.named_steps["preprocessor"]
classifier = pipeline.named_steps["classifier"]
feature_names = get_feature_names_out(preprocessor)

X_test_transformed = preprocessor.transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)
print("Transformed Test Shape:", X_test_df.shape)


## 2. Compute TreeExplainer Shapley Values


In [ ]:
explainer = shap.TreeExplainer(classifier)
shap_values = explainer(X_test_df)
print("Computed SHAP values shape:", shap_values.shape)


## 3. Global SHAP Summary Beeswarm Plot


In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("SHAP Beeswarm Summary Plot: Indian Hospital Cohort", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Mean Absolute SHAP Feature Importance Ranking


In [ ]:
mean_shap = np.abs(shap_values.values).mean(axis=0)
imp_df = pd.DataFrame({
    "feature": feature_names,
    "mean_absolute_shap": mean_shap
}).sort_values(by="mean_absolute_shap", ascending=False).reset_index(drop=True)

print("Top 10 Clinical Features by Mean |SHAP|:")
display(imp_df.head(10))

plt.figure(figsize=(8, 5))
sns.barplot(data=imp_df.head(10), x="mean_absolute_shap", y="feature", palette="Blues_r")
plt.title("Mean Absolute SHAP Feature Importance (Indian Hospital Cohort)", fontsize=12, fontweight="bold")
plt.xlabel("Mean |SHAP Value| (Impact on Log-Odds)")
plt.tight_layout()
plt.show()


## 5. Local Patient Waterfall Explanation: Confirmed High-Risk CVD Case


In [ ]:
# Select patient 0 (high risk CVD positive patient)
plt.figure(figsize=(8, 6))
shap.plots.waterfall(shap_values[0], max_display=10, show=False)
plt.title("Local Patient Explanation: Correctly Classified High-Risk Cardiac Patient", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
